![](../image/lenet.svg)
![](../image/lenet-vert.svg)

使用一个sigmoid激活函数和平均汇聚层。而非ReLU和最大汇聚层.

init_weights: xavier_uniform_

optimizer: SGD

loss: CrossEntropyLoss

In [2]:
import torch
# torchvision.datasets.FashionMNIST
import torchvision
# 修改数据集格式
from torchvision import transforms
# data.DataLoader
from torch.utils import data
# nn块
from torch import nn
# ReLU函数形式
from torch.nn import functional as F

In [15]:
# -----------参数-----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
batch_size = 16
lr = 3e-2
num_epochs=10

cuda


In [9]:
trans = transforms.ToTensor()
mnist_train_totensor = torchvision.datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=trans
)
mnist_test_totensor = torchvision.datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=trans
)
# 28*28, 不用转化大小
mnist_train_totensor[0][0].shape

torch.Size([1, 28, 28])

In [10]:
# shuffle, 打乱
# num_workers, 使用4个进程来读取数据
train_iter = data.DataLoader(
    mnist_train_totensor, batch_size, shuffle=True, num_workers=4)
test_iter = data.DataLoader(
    mnist_test_totensor, batch_size, shuffle=True, num_workers=4)

In [11]:
net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2),
    nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5),
    nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120),
    nn.Sigmoid(),
    nn.Linear(120, 84),
    nn.Sigmoid(),
    nn.Linear(84, 10)
).to(device)
net

Sequential(
  (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): Sigmoid()
  (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (4): Sigmoid()
  (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=400, out_features=120, bias=True)
  (8): Sigmoid()
  (9): Linear(in_features=120, out_features=84, bias=True)
  (10): Sigmoid()
  (11): Linear(in_features=84, out_features=10, bias=True)
)

In [12]:
def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)


net.apply(init_weights)
optimizer = torch.optim.SGD(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss()

In [19]:
def train_loop(train_iter, net, loss, optimizer):
    # 训练集总共的大小
    size = len(train_iter.dataset)
    for batch, (X, y) in enumerate(train_iter):
        # move to device
        X, y = X.to(device), y.to(device)
        # 该批的推断结果
        y_hat = net(X)
        train_loss = loss(y_hat, y)

        # Backpropagation
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            train_loss = train_loss.item()
            # 现在一共训练了的数据个数 = 批次 * 一批多少个
            current = batch * len(X)
            print(f"train_loss: {train_loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(test_iter, net, loss):
    # 测试集总共的大小
    size = len(test_iter.dataset)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in test_iter:
            # move to device
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            test_loss = loss(y_hat, y)
            correct += (y_hat.argmax(1) == y).float().sum().item()

    test_loss /= size
    correct /= size
    print(
        f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


for epoch in range(num_epochs):
    train_loop(train_iter, net, loss, optimizer)
    l = loss(net(mnist_train_totensor[:][0]), mnist_test_totensor[:][1])
    print(f'epoch {epoch + 1}, loss {l:f}')

train_loss: 2.584602  [    0/60000]
train_loss: 2.357024  [ 1600/60000]
train_loss: 2.286709  [ 3200/60000]
train_loss: 2.324796  [ 4800/60000]
train_loss: 2.320146  [ 6400/60000]
train_loss: 2.280529  [ 8000/60000]
train_loss: 2.365765  [ 9600/60000]
train_loss: 2.349147  [11200/60000]
train_loss: 2.308477  [12800/60000]
train_loss: 2.295409  [14400/60000]
train_loss: 2.285014  [16000/60000]
train_loss: 2.347470  [17600/60000]
train_loss: 2.283329  [19200/60000]
train_loss: 2.305630  [20800/60000]
train_loss: 2.269546  [22400/60000]
train_loss: 2.320883  [24000/60000]
train_loss: 2.274630  [25600/60000]
train_loss: 2.323133  [27200/60000]
train_loss: 2.305136  [28800/60000]
train_loss: 2.304900  [30400/60000]
train_loss: 2.426950  [32000/60000]
train_loss: 2.327261  [33600/60000]
train_loss: 2.308673  [35200/60000]
train_loss: 2.299212  [36800/60000]
train_loss: 2.312613  [38400/60000]
train_loss: 2.309329  [40000/60000]
train_loss: 2.378860  [41600/60000]
train_loss: 2.306043  [43200

ValueError: only one element tensors can be converted to Python scalars